In [19]:
import os

try:
    import google.colab
    REPO_URL = "https://github.com/wtheisen/nd-cse-10124-lectures.git"

    REPO_NAME = "/content/nd-cse-10124-lectures"
    L_PATH = "nd-cse-10124-lectures"

    %cd /content/
    !rm -r {REPO_NAME}

    # Clone repo
    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}

        # cd into the data folder
        %cd {L_PATH}
        !pwd

except ImportError:
    print("Unable to download repo, either:")
    print("\tA.) You're not on colab")
    print("\tB.) It has already been cloned")

%cd Lectures
!pwd
#import utilities as uts

Unable to download repo, either:
	A.) You're not on colab
	B.) It has already been cloned
/Users/wtheisen/Library/CloudStorage/GoogleDrive-wtheisen@nd.edu/My Drive/Artificial Intelligence/CSE 10124/Lectures
/Users/wtheisen/Library/CloudStorage/GoogleDrive-wtheisen@nd.edu/My Drive/Artificial Intelligence/CSE 10124/Lectures


/Users/wtheisen/Library/Python/3.9/lib/python/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [20]:
import os
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from irishGPT import tokenizer, embedding, magic_box, linear_layer

# ----------------
# Load data
# ----------------
DATA_CANDIDATES = [
    "Datasets/zoomer.txt",
]

data_path = next((p for p in DATA_CANDIDATES if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError("Could not find zoomer.txt in Datasets/ or ../Datasets/")

with open(data_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

# ----------------
# Tokenizer
# ----------------

tokenizer = tokenizer.Regex_Tokenizer()
# Use a small vocab for a quick demo
vocab_size = 512
tokenizer.train(raw_text.lower(), max_vocab_size=vocab_size)

prompt = "Hello there general kenobi"
print(tokenizer.encode(f"<|sos|>{prompt}<|eos|>"))

# ----------------
# Dataset + DataLoader
# ----------------

class TextDataset(Dataset):
    def __init__(self, text, tokenizer, seq_len=32, device="cpu"):
        self.seq_len = seq_len
        self.device = device
        self.padding_idx = -1  # no padding in this dataset

        tokens = []
        for line in text.splitlines():
            line = line.strip()
            if not line:
                continue
            line = f"<|sos|>{line}<|eos|>"
            tokens.extend(tokenizer.encode(line.lower()))

        if len(tokens) < seq_len + 1:
            raise ValueError("Text is too short for the chosen seq_len")

        self.tokens = torch.tensor(tokens, dtype=torch.long, device=device)
        self.vocab_size = max(tokenizer.vocab.keys()) + 1

    def __len__(self):
        return len(self.tokens) - self.seq_len - 1

    def __getitem__(self, idx):
        x = self.tokens[idx : idx + self.seq_len]
        y = self.tokens[idx + 1 : idx + self.seq_len + 1]
        return x, y

device = torch.device('cuda' if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

seq_len = 32
batch_size = 64

dataset = TextDataset(raw_text, tokenizer, seq_len=seq_len, device=device)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


[256, 72, 101, 291, 111, 274, 276, 281, 298, 290, 329, 377, 298, 111, 98, 105, 257]


In [ ]:
from typing import Any
import torch
import numpy as np
from irishGPT import embedding, magic_box, linear_layer

class SmallLanguageModel:
    def __init__(self, vocab_size, embed_size, hidden_size, padding_idx=0, device='cpu'):
        self.vocab_size = vocab_size
        self.embed_size = embed_size
        self.hidden_size = hidden_size
        self.padding_idx = padding_idx
        self.device = device

        # Layers:
        embedding_layer = embedding.EmbeddingLayer(vocab_size, embed_size, device=device)
        # For the recurrent context layer, the input dimension is the embed_size.
        recurrent_block = magic_box.MagicBox(embed_size, hidden_size, device=device)
        # Final fully connected layer: project hidden state to vocabulary logits.
        output_layer = linear_layer.LinearLayer(hidden_size, vocab_size, device=device)

        # Keep model layers in a list for easy backward and update passes.
        self.layers = [embedding_layer, recurrent_block, output_layer]

    def forward(self, X, eval=False):
        """
        Args:
          x: (batch_size, seq_len) with integer token indices.
        Returns:
          logits: (batch_size, seq_len, vocab_size)
        """
        for layer in self.layers:
            X = layer.forward(X)

        return X if not eval else self.softmax(X)

    def softmax(self, X):
        """
        Args:
            X (torch.Tensor): Input data with shape (..., n_classes)

        Returns:
            torch.Tensor: Softmax probabilities with shape (..., n_classes)
        """
        m = torch.amax(X, dim=-1, keepdim=True)
        ex = torch.exp(X - m)
        return ex / ex.sum(dim=-1, keepdim=True)

    def cross_entropy(self, logits, Y_idx):
        """
        Compute the cross-entropy loss from logits and class indices.

        Args:
            logits (torch.Tensor): (B, T, V)
            Y_idx (torch.Tensor): (B, T) integer class indices
        """
        log_probs = torch.log(self.softmax(logits).clamp_min(1e-12))
        token_loss = -log_probs.gather(dim=-1, index=Y_idx.unsqueeze(-1)).squeeze(-1)  # (B, T)
        mask = (Y_idx != self.padding_idx).float()
        masked_loss = token_loss * mask
        normalizer = mask.sum().clamp_min(1.0)
        return masked_loss.sum() / normalizer

    def get_accuracy(self, logits, Y_idx):
        preds = logits.argmax(dim=-1)
        mask = (Y_idx != self.padding_idx).float()
        correct = (preds == Y_idx).float() * mask
        normalizer = mask.sum().clamp_min(1.0)
        return correct.sum() / normalizer

    def backward(self, logits, Y_idx):
        probs = self.softmax(logits)
        # One-hot inside for gradient only
        Y_onehot = torch.nn.functional.one_hot(Y_idx, num_classes=self.vocab_size).float()
        mask = (Y_idx != self.padding_idx).unsqueeze(-1).float()
        normalizer = mask.sum().clamp_min(1.0)
        dA = (probs - Y_onehot) * mask / normalizer

        for layer in reversed(self.layers):
            dA = layer.backward(dA)

    def generate(self, tokenizer, prompt, max_new_tokens=50, temperature=1.0, top_k=None, device="cpu"):
        model_prompt = f"<|sos|>{prompt}"
        ids = tokenizer.encode(model_prompt.lower())

        with torch.no_grad():
            for _ in range(max_new_tokens):
                x = torch.tensor([ids], dtype=torch.long, device=device)
                logits = self.forward(x)[:, -1, :]  # (1, vocab)
                logits = logits / max(1e-6, temperature)
                print('Token probabilities:', logits)

                if top_k is not None and top_k > 0:
                    topv, topi = torch.topk(logits, k=min(top_k, logits.size(-1)), dim=-1)
                    probs = torch.softmax(topv, dim=-1)
                    next_id = topi[0, torch.multinomial(probs, 1).item()].item()
                else:
                    probs = torch.softmax(logits, dim=-1)
                    next_id = torch.multinomial(probs, 1).item()

                ids.append(next_id)
                print(ids)
                if next_id == tokenizer.special_tokens.get("<|eos|>"):
                    break

        text = tokenizer.decode(ids)
        return text.replace("<|sos|>", "").replace("<|eos|>", "")

    def train(self, train_loader, epochs=10, learning_rate=0.05, tokenizer=None, verbose=True):
        loss_history = []
        accuracy_history = []
        generation_snapshots = {}

        if verbose:
            import matplotlib.pyplot as plt
            from IPython.display import clear_output

        for i in range(epochs):
            batch_losses = []
            batch_accuracies = []

            for X_batch, Y_batch in train_loader:
                # Forward propagation
                logits = self.forward(X_batch)

                # Calculate metrics for the whole epoch
                loss = self.cross_entropy(logits, Y_batch)
                accuracy = self.get_accuracy(logits, Y_batch)

                batch_losses.append(loss.item())
                batch_accuracies.append(accuracy.item())

                # Backward propagation
                self.backward(logits, Y_batch)

                # Update parameters
                for layer in self.layers:
                    layer.update(learning_rate)

            loss_history.append(np.mean(batch_losses))
            accuracy_history.append(np.mean(batch_accuracies))

            if verbose:
                if i % 25 == 0:
                    generation_snapshots[i] = self.generate(tokenizer, "I", device=device)
                    print(f"Generation at epoch {i}: I {generation_snapshots[i]}")
                # clear_output(wait=True)
                # print(f"Epoch {i+1}/{epochs}")
                # print(f"loss: {loss_history[-1]:.5f}")
                # print(f"accuracy: {accuracy_history[-1]:.5f}")
                # print("-" * 30)

                # plt.figure(figsize=(7, 4))
                # plt.plot(loss_history, label='loss')
                # plt.plot(accuracy_history, label='accuracy')
                # plt.xlabel('epoch')
                # plt.legend()
                # plt.title('Training Curves')
                # plt.grid(True)
                # plt.show()
                # plt.close()

        return {'loss_history': loss_history, 'accuracy_history': accuracy_history, 'generation_snapshots': generation_snapshots}


In [25]:
SLM = SmallLanguageModel(vocab_size=dataset.vocab_size,  embed_size=64, hidden_size=128, padding_idx=dataset.padding_idx, device=device)

history = SLM.train(train_loader, tokenizer=tokenizer, epochs=250, verbose=True)

for epoch, prompt in history['generation_snapshots'].items():
    print(f"Generation at epoch {epoch}: I {prompt}")


Generation at epoch 0: I ] finis� rf a out s.
Generation at epoch 25: I you cend is ag week,ly lo lo!
Generation at epoch 50: I that tomel was a qase, jci thious has right
Generation at epoch 75: I that new seeig was alwrapse, how see night.
Generation at epoch 100: I she love this wko that spute.
Generation at epoch 125: I let’s from the comporit is so bol pi.
Generation at epoch 150: I i’ll letking here-arluth.
Generation at epoch 175: I let’s not talking about that movie? salik
Generation at epoch 200: I cancew the meeting think it rixar.
Generation at epoch 225: I na finish the new tell me! dknt.
Generation at epoch 0: I ] finis� rf a out s.
Generation at epoch 25: I you cend is ag week,ly lo lo!
Generation at epoch 50: I that tomel was a qase, jci thious has right
Generation at epoch 75: I that new seeig was alwrapse, how see night.
Generation at epoch 100: I she love this wko that spute.
Generation at epoch 125: I let’s from the comporit is so bol pi.
Generation at epoch 150: I i

In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=50, temperature=1.0, top_k=20, device="cpu"):
    model_prompt = f"<|sos|>{prompt}"
    ids = tokenizer.encode(model_prompt.lower())

    with torch.no_grad():
        for _ in range(max_new_tokens):
            x = torch.tensor([ids], dtype=torch.long, device=device)
            logits = model.forward(x)[:, -1, :]  # (1, vocab)
            logits = logits / max(1e-6, temperature)

            if top_k is not None and top_k > 0:
                topv, topi = torch.topk(logits, k=min(top_k, logits.size(-1)), dim=-1)
                probs = torch.softmax(topv, dim=-1)
                next_id = topi[0, torch.multinomial(probs, 1).item()].item()
            else:
                probs = torch.softmax(logits, dim=-1)
                next_id = torch.multinomial(probs, 1).item()

            ids.append(next_id)
            if next_id == tokenizer.special_tokens.get("<|eos|>"):
                break

    text = tokenizer.decode(ids)
    return text.replace("<|sos|>", "").replace("<|eos|>", "")

prompt = "I"
print(prompt, generate(SLM, tokenizer, prompt, max_new_tokens=60, temperature=0.9, top_k=30, device=device))


## Export to HTML

Uncomment the final line of the cell below and run it to export this notebook to HTML

In [ ]:
import os, json

def export_notebook():
  L_PATH = "nd-cse-10124-lectures/Notebooks"
  L = "Lecture_07_Embeddings_02"

  try:
      from google.colab import _message, files

      # where you WANT it to live (repo folder)
      repo_ipynb_path = f"/content/{L_PATH}/{L}.ipynb"

      # grab current notebook contents from the UI
      nb = _message.blocking_request("get_ipynb", timeout_sec=1)["ipynb"]

      # write it into the repo folder as a real file
      os.makedirs(os.path.dirname(repo_ipynb_path), exist_ok=True)
      with open(repo_ipynb_path, "w", encoding="utf-8") as f:
          json.dump(nb, f)

      # convert + download pdf
      !jupyter nbconvert --to html "{repo_ipynb_path}"
      files.download(repo_ipynb_path.replace(".ipynb", ".html"))
  except:
      import subprocess

      nb_fp = os.getcwd() + f'{L}.ipynb'
      print(os.getcwd())

      subprocess.run(["jupyter", "nbconvert", "--to", "html", nb_fp], check=True)

#export_notebook()